In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import Phylo
import sys
sys.path.append('../pysimARG')
from clonal_genealogy import ClonalTree
from ClonalOrigin_seq_sim import ClonalOrigin_seq_sim
from localtree_simbac import load_local_trees, get_tree_at_position
from homoplasy_index_simbac import homoplasy_index_simbac
from newick_to_tree import newick_to_tree
from G4_test import G4_test
from LD_r import LD_r

In [3]:
np.random.seed(100)
tree = ClonalTree(n=10)

# Load phylo tree and convert to ClonalTree format
phylo_tree = Phylo.read("../data/SimBac/clonal_frame.nwk", "newick")
Phylo.draw_ascii(phylo_tree)

edge, node_height = newick_to_tree(phylo_tree)
tree.edge = edge
tree.node_height = node_height
tree.height = np.max(node_height)
tree.length = np.sum(edge[:, 2])

rho_site = 0.02
theta_site = 0.05
L = 2000
delta = 30

                                                                   _______ 2
                                        __________________________|
                            ___________|                          |_______ 8
                           |           |
                           |           |___________________________________ 1
                           |
                        ___|                         _____________________ 6
                       |   |                       ,|
                       |   |                       ||  ___________________ 3
                       |   |                       ||_|
                       |   |_______________________|  |___________________ 9
  _____________________|                           |
 |                     |                           |         _____________ 5
 |                     |                           |________|
_|                     |                                    |_____________ 7
 |                  

## SimBac summary stats

In [2]:
genome_simbac = np.loadtxt("../data/SimBac/genomes_bool.csv", delimiter=",", dtype=bool)
genome_simbac.shape

(10, 100000)

In [4]:
file_path = "../data/SimBac/local_tree.nwk"

blocks = load_local_trees(file_path)

print(f"\nSuccessfully loaded {len(blocks)} local trees.")

print("\n--- Genome Map Overview ---")
current_position = 1

for i, (span, tree) in enumerate(blocks[:5]):
    end_position = current_position + span - 1
    print(f"Block {i+1}: Sites {current_position:05d} to {end_position:05d} (Length: {span}bp) - Tree has {len(tree.get_terminals())} leaves")
    current_position += span

Parsing local trees...

Successfully loaded 10618 local trees.

--- Genome Map Overview ---
Block 1: Sites 00001 to 00025 (Length: 25bp) - Tree has 10 leaves
Block 2: Sites 00026 to 00060 (Length: 35bp) - Tree has 10 leaves
Block 3: Sites 00061 to 00088 (Length: 28bp) - Tree has 10 leaves
Block 4: Sites 00089 to 00119 (Length: 31bp) - Tree has 10 leaves
Block 5: Sites 00120 to 00123 (Length: 4bp) - Tree has 10 leaves


In [ ]:
simbac_stats = np.full((1000, 6), np.nan)
simbac_stats

array([[nan, nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan, nan],
       ...,
       [nan, nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan, nan]], shape=(10000, 6))

In [ ]:
np.random.seed(100)
for idx in range(1000):
    start_pos = np.random.randint(1, genome_simbac.shape[1]-L+1)
    mat = genome_simbac[:, start_pos-1:start_pos+L-1]

    has_true = mat.any(axis=0)
    has_false = ~mat.all(axis=0)
    idx_seg = np.where(has_true & has_false)[0]

    ld_near, ld_far, g4_near, g4_far = 0, 0, 0, 0
    if idx_seg.size >= 2:
        for i in range(idx_seg.size - 1):
            for j in range(i + 1, idx_seg.size):
                dist_ij = idx_seg[j] - idx_seg[i]
                idx_pair = [idx_seg[i], idx_seg[j]]
                if dist_ij < L/2:
                    ld_near += LD_r(mat[:, idx_pair])
                    g4_near += G4_test(mat[:, idx_pair])
                else:
                    ld_far += LD_r(mat[:, idx_pair])
                    g4_far += G4_test(mat[:, idx_pair])
        
        s_far = (int(L/2) + 1) * (int(L/2)) / 2
        s_near = L * (L - 1) / 2 - s_far
        simbac_stats[idx, 0] = ld_near / s_near
        simbac_stats[idx, 1] = ld_far / s_far
        simbac_stats[idx, 2] = g4_near / s_near
        simbac_stats[idx, 3] = g4_far / s_far
    else:
        simbac_stats[idx, :4] = 0
    
    simbac_stats[idx, 4] = homoplasy_index_simbac(blocks, genome_simbac, start_pos, L)
    count_S = idx_seg.size
    simbac_stats[idx, 5] = count_S / L

    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1} segments...")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
